In [0]:
# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Gold
# Notebook        : Gold_Fact_Purchase_Requisitions
# Source          : Purchase_Requisitions.csv
# Target          : procurement.gold.Fact_Purchase_Requisitions

#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook created the Fact_Purchase_Requisitions table in the gold layer.
#
# ==============================================================================
# Business Objective
# ==============================================================================
# Create the Purchase_Requisitions Fact table from the Silver layer.
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
# ============================================================
# Read Silver Purchase_Requisitions Table
# ============================================================

silver_Purchase_Requisitions_df = read_delta(SILVER_PURCHASE_REQUISITIONS)

preview(silver_Purchase_Requisitions_df,"Silver Purchase_Requisitions")

In [0]:

from pyspark.sql import functions as F

# ============================================================
# 1. Reporting Date
# ============================================================

AS_OF_DATE = "2025-12-31"


# ============================================================
# 2. Read Silver Tables
# ============================================================

silver_Purchase_Requisitions_df = read_delta(
    SILVER_PURCHASE_REQUISITIONS
)

silver_Employees_df = read_delta(SILVER_EMPLOYEES)

silver_Department_df = read_delta(SILVER_DEPARTMENTS)

silver_Products_df = read_delta(SILVER_PRODUCTS)

silver_Purchase_Orders_df = read_delta(
    SILVER_PURCHASE_ORDERS
)


# ============================================================
# 3. Prepare Dimension Tables
# ============================================================

employees_df = silver_Employees_df.select(
    "employee_id",
    "employee_name",
    "job_title",
    "is_active"
)

department_df = silver_Department_df.select(
    "department_id",
    "department_name",
    "division",
    "region",
    "cost_center",
    "annual_budget_usd"
)

products_df = silver_Products_df.select(
    "product_id",
    "product_name",
    "category",
    "unit_of_measure",
    "standard_unit_price",
    "currency"
)


# ============================================================
# 4. Summarize POs: One Row Per PR
# ============================================================

po_summary_df = (
    silver_Purchase_Orders_df
    .groupBy("pr_id")
    .agg(
        F.countDistinct("po_id").alias("total_pos"),

        F.sum("total_amount").alias(
            "total_po_amount"
        ),

        F.sum("quantity_ordered").alias(
            "total_quantity_ordered"
        ),

        F.min("po_date").alias("first_po_date"),

        F.max("po_date").alias("latest_po_date"),

        F.min("expected_delivery_date").alias(
            "first_expected_delivery_date"
        ),

        F.sum(
            F.when(
                F.col("is_duplicate_po") == True, 1
            ).otherwise(0)
        ).alias("duplicate_po_count")
    )
)


# ============================================================
# 5. Join PR With Dimensions and PO Summary
# ============================================================

gold_base_df = (
    silver_Purchase_Requisitions_df.alias("pr")

    .join(
        employees_df.alias("e"),
        F.col("pr.employee_id") == F.col("e.employee_id"),
        "left"
    )

    .join(
        department_df.alias("d"),
        F.col("pr.department_id") == F.col("d.department_id"),
        "left"
    )

    .join(
        products_df.alias("p"),
        F.col("pr.product_id") == F.col("p.product_id"),
        "left"
    )

    .join(
        po_summary_df.alias("po"),
        F.col("pr.pr_id") == F.col("po.pr_id"),
        "left"
    )
)


# ============================================================
# 6. Create Gold PR-Level Columns
# ============================================================

gold_fact_Purchase_Requisitions_df = (
    gold_base_df

    # Replace missing PO summary values with zero
    .withColumn(
        "total_pos",
        F.coalesce(F.col("total_pos"), F.lit(0))
    )

    .withColumn(
        "total_po_amount",
        F.coalesce(
            F.col("total_po_amount"),
            F.lit(0).cast("decimal(12,2)")
        )
    )

    .withColumn(
        "total_quantity_ordered",
        F.coalesce(
            F.col("total_quantity_ordered"),
            F.lit(0)
        )
    )

    .withColumn(
        "duplicate_po_count",
        F.coalesce(
            F.col("duplicate_po_count"),
            F.lit(0)
        )
    )

    # PR age
    .withColumn(
        "pr_age_days",
        F.datediff(
            F.lit(AS_OF_DATE).cast("date"),
            F.col("pr.pr_date")
        )
    )

    # Approval turnaround
    .withColumn(
        "approval_days",
        F.when(
            F.col("pr.approval_date").isNotNull(),
            F.datediff(
                F.col("pr.approval_date"),
                F.col("pr.pr_date")
            )
        )
    )

    # Approval SLA: 3 days
    .withColumn(
        "approval_sla_status",
        F.when(
            F.col("pr.approval_date").isNull(),
            "Pending Approval"
        )
        .when(
            F.datediff(
                F.col("pr.approval_date"),
                F.col("pr.pr_date")
            ) <= 3,
            "Within SLA"
        )
        .otherwise("SLA Breached")
    )

    # PR to PO creation status
    .withColumn(
        "po_creation_status",
        F.when(
            F.col("total_pos") == 0,
            "PO Not Created"
        ).otherwise("PO Created")
    )

    # Days from PR creation to first PO
    .withColumn(
        "pr_to_po_days",
        F.when(
            F.col("first_po_date").isNotNull(),
            F.datediff(
                F.col("first_po_date"),
                F.col("pr.pr_date")
            )
        )
    )

    # PR to first PO turnaround status
    .withColumn(
        "pr_to_po_status",
        F.when(
            F.col("total_pos") == 0,
            "PO Not Created"
        )
        .when(
            F.col("pr_to_po_days") <= 3,
            "Fast"
        )
        .when(
            F.col("pr_to_po_days") <= 7,
            "Normal"
        )
        .otherwise("Delayed")
    )

    # Combined quantity variance across all POs
    .withColumn(
        "quantity_variance",
        F.when(
            F.col("total_pos") > 0,
            F.col("total_quantity_ordered")
            - F.col("pr.quantity_requested")
        )
    )

    # Combined quantity fulfillment percentage
    .withColumn(
        "quantity_fulfillment_pct",
        F.when(
            F.col("pr.quantity_requested") > 0,
            F.round(
                F.col("total_quantity_ordered")
                / F.col("pr.quantity_requested") * 100,
                2
            )
        )
    )

    # Combined PO value compared with PR estimate
    .withColumn(
        "po_value_variance",
        F.when(
            F.col("total_pos") > 0,
            F.col("total_po_amount")
            - F.col("pr.estimated_total")
        )
    )

    .withColumn(
        "po_value_variance_pct",
        F.when(
            (F.col("pr.estimated_total") > 0)
            & (F.col("total_pos") > 0),
            F.round(
                (
                    F.col("total_po_amount")
                    - F.col("pr.estimated_total")
                )
                / F.col("pr.estimated_total") * 100,
                2
            )
        )
    )

    # Compare PR estimated unit price with standard price
    .withColumn(
        "unit_price_variance",
        F.when(
            F.col("p.standard_unit_price").isNotNull(),
            F.col("pr.estimated_unit_price")
            - F.col("p.standard_unit_price")
        )
    )

    .withColumn(
        "unit_price_variance_pct",
        F.when(
            F.col("p.standard_unit_price") > 0,
            F.round(
                (
                    F.col("pr.estimated_unit_price")
                    - F.col("p.standard_unit_price")
                )
                / F.col("p.standard_unit_price") * 100,
                2
            )
        )
    )

    # Price control
    .withColumn(
        "price_control_status",
        F.when(
            F.col("p.standard_unit_price").isNull(),
            "No Standard Price"
        )
        .when(
            F.col("pr.estimated_unit_price")
            > F.col("p.standard_unit_price"),
            "Above Standard Price"
        )
        .when(
            F.col("pr.estimated_unit_price")
            < F.col("p.standard_unit_price"),
            "Below Standard Price"
        )
        .otherwise("At Standard Price")
    )

    # PR lifecycle
    .withColumn(
        "pr_lifecycle_status",
        F.when(
            F.col("pr.status") == "Rejected",
            "Rejected"
        )
        .when(
            F.col("pr.status") == "Cancelled",
            "Cancelled"
        )
        .when(
            F.col("pr.approval_date").isNull(),
            "Pending Approval"
        )
        .when(
            F.col("total_pos") == 0,
            "Approved - PO Not Created"
        )
        .otherwise("Approved - PO Created")
    )

    .withColumn(
        "gold_load_timestamp",
        F.current_timestamp()
    )
)


# ============================================================
# 7. Select Final Gold Columns
# ============================================================

gold_fact_Purchase_Requisitions_df = (
    gold_fact_Purchase_Requisitions_df
    .select(
        # PR information
        F.col("pr.pr_id").alias("pr_id"),
        F.col("pr.pr_date").alias("pr_date"),
        F.col("pr.employee_id").alias("employee_id"),
        F.col("e.employee_name").alias("employee_name"),
        F.col("e.job_title").alias("job_title"),
        F.col("e.is_active").alias("employee_is_active"),

        F.col("pr.department_id").alias("department_id"),
        F.col("d.department_name").alias("department_name"),
        F.col("d.division").alias("division"),
        F.col("d.region").alias("region"),
        F.col("d.cost_center").alias("cost_center"),
        F.col("d.annual_budget_usd").alias("annual_budget_usd"),

        F.col("pr.product_id").alias("product_id"),
        F.col("p.product_name").alias("product_name"),
        F.col("p.category").alias("product_category"),
        F.col("p.unit_of_measure").alias("unit_of_measure"),
        F.col("p.standard_unit_price").alias("standard_unit_price"),
        F.col("p.currency").alias("product_currency"),

        F.col("pr.quantity_requested").alias("quantity_requested"),
        F.col("pr.estimated_unit_price").alias("estimated_unit_price"),
        F.col("pr.estimated_total").alias("estimated_total"),
        F.col("pr.priority").alias("priority"),
        F.col("pr.status").alias("status"),
        F.col("pr.approver_id").alias("approver_id"),
        F.col("pr.approval_date").alias("approval_date"),

        # PO summary information
        F.col("total_pos"),
        F.col("total_po_amount"),
        F.col("total_quantity_ordered"),
        F.col("first_po_date"),
        F.col("latest_po_date"),
        F.col("first_expected_delivery_date"),
        F.col("duplicate_po_count"),

        # Calculated metrics
        F.col("pr_age_days"),
        F.col("approval_days"),
        F.col("approval_sla_status"),
        F.col("po_creation_status"),
        F.col("pr_to_po_days"),
        F.col("pr_to_po_status"),
        F.col("quantity_variance"),
        F.col("quantity_fulfillment_pct"),
        F.col("po_value_variance"),
        F.col("po_value_variance_pct"),
        F.col("unit_price_variance"),
        F.col("unit_price_variance_pct"),
        F.col("price_control_status"),
        F.col("pr_lifecycle_status"),
        F.col("gold_load_timestamp")
    )
)


# ============================================================
# 8. Preview Gold Data
# ============================================================

print(
    "Gold PR rows:",
    gold_fact_Purchase_Requisitions_df.count()
)

print(
    "Distinct Gold PR IDs:",
    gold_fact_Purchase_Requisitions_df
    .select("pr_id")
    .distinct()
    .count()
)

display(
    gold_fact_Purchase_Requisitions_df.limit(10)
)


In [0]:
# ============================================================
# Add Gold Metadata
# ============================================================

gold_fact_Purchase_Requisitions_df = (
    gold_fact_Purchase_Requisitions_df
        .withColumn(
            "gold_load_timestamp",
            current_timestamp()
        )
)

In [0]:
# ============================================================
# Write Gold Delta Table
# ============================================================

write_delta(
    df=gold_fact_Purchase_Requisitions_df,
    table_name=GOLD_FACT_PURCHASE_REQUISITIONS
)

In [0]:
# ============================================================
# Validate Gold Delta Table
# ============================================================

validate_table(GOLD_FACT_PURCHASE_REQUISITIONS)

In [0]:
# ============================================================
# Gold Layer Summary
# ============================================================

print("=" * 60)
print("Gold Payments Load Completed Successfully")
print("=" * 60)

print(f"Silver Records : {silver_Purchase_Requisitions_df.count():,}")
print(f"Gold Records   : {gold_fact_Purchase_Requisitions_df.count():,}")